In [ ]:
import sys, glob, json, subprocess, traceback, time
from pathlib import Path
hits = sorted(glob.glob('/kaggle/input/**/shared_code.py', recursive=True))
W = Path(hits[0]).parent
sys.path.insert(0, str(W))
tr = sorted(glob.glob('/kaggle/input/**/train_series.csv', recursive=True))
COMP = Path(tr[0]).parent
print('weights:', W, '| comp:', COMP)
print('mounted:', sorted(p.name for p in COMP.iterdir()))

In [ ]:
wheels = sorted(glob.glob(str(W / '*.whl')))
try:
    import pylibjpeg, openjpeg, libjpeg
    print('decoders already present')
except ImportError:
    if wheels:
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index',
                            '--no-deps', '-q', *wheels], capture_output=True, text=True)
        print('installed decoders rc=', r.returncode, r.stderr[-300:])
import pydicom
print('pydicom', pydicom.__version__)
from pydicom.uid import JPEG2000Lossless, JPEGLosslessSV1
for uid in (JPEG2000Lossless, JPEGLosslessSV1):
    try:
        import pydicom.pixels as px
        print(uid.name, '->', [d for d in px.decoders.get_decoder(uid).available_plugins])
    except Exception as e:
        print(uid.name, 'probe failed:', type(e).__name__, e)

In [ ]:
import shared_code as sc, pandas as pd, torch, numpy as np
cfg = json.loads((W / 'config.json').read_text())
cfg['model']['pretrained'] = False
TASKS = cfg['tasks']
series = pd.read_csv(COMP / 'train_series.csv')
labels = pd.read_csv(COMP / 'train.csv')
print('train studies', labels.shape, '| series', series.shape)

# how many transfer syntaxes are actually in this corpus? this is the question
# that decided the last three submissions and we have never once measured it.
import collections
syn = collections.Counter()
files = []
root = COMP / 'train_series'
for st in sorted(p for p in root.iterdir() if p.is_dir())[:60]:
    for se in sorted(p for p in st.iterdir() if p.is_dir()):
        fs = sorted(se.iterdir())
        if fs:
            files.append(fs[0])
for f in files:
    try:
        ds = pydicom.dcmread(str(f), stop_before_pixels=True, force=True)
        syn[str(getattr(ds.file_meta, 'TransferSyntaxUID', 'NONE'))] += 1
    except Exception as e:
        syn['READ FAILED ' + type(e).__name__] += 1
for k, v in syn.most_common():
    try:
        from pydicom.uid import UID
        nm = UID(k).name
    except Exception:
        nm = k
    print(f'{v:6d}  {nm}')

In [ ]:
# now decode one slice of each of those series for real, and count what breaks
ok = bad = 0
errs = collections.Counter()
t0 = time.time()
for f in files:
    try:
        ds = pydicom.dcmread(str(f), force=True)
        _ = ds.pixel_array
        ok += 1
    except Exception as e:
        bad += 1
        errs[f'{type(e).__name__}: {str(e)[:90]}'] += 1
print(f'decoded {ok}/{ok + bad} first-slices in {time.time() - t0:.0f}s')
for k, v in errs.most_common(10):
    print(f'  {v:5d}  {k}')

In [ ]:
# the real test path, exactly as the submit notebook runs it
tser = pd.read_csv(COMP / 'test_series.csv')
tsub = pd.read_csv(COMP / 'sample_submission.csv')
print('test_series rows', len(tser), '| studies in sample_submission', len(tsub))
print('studies in test_series.csv :', tser.StudyInstanceUID.nunique())
missing_dirs = [s for s in tser.StudyInstanceUID.astype(str).unique()
                if not (COMP / 'test_series' / s).is_dir()]
print('studies with NO folder     :', len(missing_dirs))
empty = 0
for _, r in tser.iterrows():
    d = COMP / 'test_series' / str(r.StudyInstanceUID) / str(r.SeriesInstanceUID)
    if not d.is_dir() or not any(d.iterdir()):
        empty += 1
print('series dirs missing/empty  :', empty)
t0 = time.time()
tidx, tfbs = sc.scan_dicom_index(COMP / 'test_series', tser)
print(f'scan {time.time() - t0:.0f}s -> {len(tidx)} of {len(tser)} series indexed')
tplan = sc.plan_slots(tidx, cfg)
print(len(tplan), 'studies planned of', len(tsub))